In [21]:
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
print("📥 Carregando e processando dados...")
inicio = time.time()
# carga de arquivo XLSX demora muito (pelos testes, em torno de 1 minuto),  então vamos usar CSV para agilizar o processo
#df = pd.read_excel("dataset/Focos-AmazoniaLegal2020-2025-Final.xlsx")
df = pd.read_csv("/content/drive/MyDrive/Dataset/Focos-AmazoniaLegal2020-2025-Final.csv", encoding="utf-8", sep=";", low_memory=False)
print(f"✅ Dados carregados com sucesso!\r\nTotal de registros: {len(df)}\r\nTempo: {time.time() - inicio:.2f} segundos")


📥 Carregando e processando dados...
✅ Dados carregados com sucesso!
Total de registros: 787709
Tempo: 3.48 segundos


In [24]:
print("📊 Processando dados...")
inicio = time.time()

df = df.dropna(subset=["FRP"])

df["Mes"] = df["Mes"].astype(int)
df["DiaSemChuva"] = df["DiaSemChuva"].astype(float)
df["Precipitacao"] = df["Precipitacao"].astype(float)

print(f"✅ Dados limpos!\r\nTotal de registros: {len(df)}\r\nTempo: {time.time() - inicio:.2f} segundos")


📊 Processando dados...
✅ Dados limpos!
Total de registros: 785440
Tempo: 0.13 segundos


In [25]:
def converter_nome_para_sigla(df, coluna_estado="Estado"):
    """
    Mapeia os nomes cheios dos estados da Amazônia Legal para suas respectivas siglas.
    """
    mapeamento_uf = {
        "ACRE": "AC",
        "AMAPÁ": "AP",
        "AMAZONAS": "AM",
        "MARANHÃO": "MA",
        "MATO GROSSO": "MT",
        "PARÁ": "PA",
        "RONDÔNIA": "RO",
        "RORAIMA": "RR",
        "TOCANTINS": "TO"
    }

    df[coluna_estado] = df[coluna_estado].astype(str).str.strip()
    df[coluna_estado] = df[coluna_estado].map(mapeamento_uf).fillna(df[coluna_estado])

    return df

def add_features(df):
    def estacao(mes):
        if mes in [11, 12, 1, 2, 3, 4, 5]:
            return "chuvosa"
        return "seca"

    df["Estacao"] = df["Mes"].apply(estacao)
    df = converter_nome_para_sigla(df, coluna_estado="Estado")
    df["Municipio_UF"] = df["Municipio"].astype(str) + " - " + df["Estado"].astype(str)

    return df

In [26]:
print("🔄 Adicionando características...")
inicio = time.time()
df = add_features(df)
print(f"✅ Características adicionadas!\r\nTempo: {time.time() - inicio:.2f} segundos")

🔄 Adicionando características...
✅ Características adicionadas!
Tempo: 0.67 segundos


In [27]:
# alvo
df = df[df["FRP"] > 0]

y = np.log1p(df["FRP"])

In [28]:
# features
numeric_features = ["Mes", "DiaSemChuva", "Precipitacao"]
categorical_features = ["Municipio_UF", "Estacao"]

X = df[numeric_features + categorical_features]


In [29]:
# preprocessamento
preprocessador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [30]:
# modelos
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

gb = GradientBoostingRegressor(random_state=42)

rf_model = Pipeline([
    ("prep", preprocessador),
    ("model", rf)
])

gb_model = Pipeline([
    ("prep", preprocessador),
    ("model", gb)
])


In [31]:
# divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [32]:
print("🔄Treinando Random Forest...")
inicio2 = time.time()
rf_model.fit(X_train, y_train)
print("✅Random Forest treinado!\r\n Tempo: {:.2f} segundos".format(time.time() - inicio2))


🔄Treinando Random Forest...
✅Random Forest treinado!
 Tempo: 531.51 segundos


In [33]:
print("🔄Treinando Gradient Boosting...")
inicio2 = time.time()
gb_model.fit(X_train, y_train)
print("Gradient Boosting treinado!\r\n Tempo: {:.2f} segundos".format(time.time() - inicio2))


🔄Treinando Gradient Boosting...
Gradient Boosting treinado!
 Tempo: 62.64 segundos


In [34]:
# função de avaliação dos modelos
def avaliar_modelo(nome, modelo):

    preds = modelo.predict(X_test)

    mae = mean_absolute_error(y_test, preds)

    rmse = np.sqrt(
        mean_squared_error(y_test, preds)
    )

    r2 = r2_score(y_test, preds)

    print(f"\n📊 {nome}")
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R²:", round(r2, 4))

    # Retorna as métricas como um dicionário
    return {"Modelo": nome, "MAE": round(mae, 4), "RMSE": round(rmse, 4), "R²": round(r2, 4)}

In [35]:
linhas_metricas = []
linhas_metricas.append(avaliar_modelo("Random Forest", rf_model))
linhas_metricas.append(avaliar_modelo("Gradient Boosting", gb_model))


📊 Random Forest
MAE: 0.7368
RMSE: 0.932
R²: 0.1074

📊 Gradient Boosting
MAE: 0.7442
RMSE: 0.9411
R²: 0.0898


In [36]:
# Salva os modelos em disco
joblib.dump(rf_model, "/content/drive/MyDrive/Dataset/rf_model.pkl")
joblib.dump(gb_model, "/content/drive/MyDrive/Dataset/gb_model.pkl")

['/content/drive/MyDrive/Dataset/gb_model.pkl']

In [37]:
# salvar municípios
municipios = sorted(df["Municipio_UF"].unique())
joblib.dump(municipios, "/content/drive/MyDrive/Dataset/municipios.pkl")

df_metricas = pd.DataFrame(linhas_metricas)
df_metricas.to_csv("/content/drive/MyDrive/Dataset/metricas.csv", index=False)

# dados para o gráfico de focos por mês
grafico_df = (
    df.groupby(["Municipio_UF", "Mes"])
      .size()
      .reset_index(name="QuantidadeFocos")
)

grafico_df.to_csv(
    "/content/drive/MyDrive/Dataset/focos_por_mes.csv",
    index=False
)

print("\n✔ Modelos e arquivos salvos!")


✔ Modelos e arquivos salvos!
